# 3. Prepare OAI Data

This notebook builds the full `(X, y)` dataset from raw OAI DICOM files and the semi-quantitative label file.

**Pipeline:**
1. Walk `ROOT_DIR` → find all DICOM `001` files → filter to `BodyPartExamined == KNEE`
2. Inner join with `oai_kxrsemiquant01.txt` on `(src_subject_id, interview_date)` → one row per **side** (left/right)
3. Preprocess each bilateral DICOM using `XRayImageDataset` methods from pain-disparities:
   - resize to 1024×1024 → divide by dataset max → `cut_image_in_half()` → z-score → save as `.npy`
4. Process labels using `NonImageData.load_semiquantitative_xray_data()` logic from pain-disparities:
   - truncate JSN, fill concepts=0 for never-KLG≥2, merge KLG classes
5. Save `manifest.csv` (all metadata + labels) and `stats.json` (normalisation constants)

**Re-runnable:** just re-run after downloading more data — new images are processed, already-processed images are skipped.

## Config

In [1]:
import os

REPO_DIR      = "/Users/yuhaohe/Documents/RA/healthcare_AI_RA/ConceptBottleneck"

# Raw DICOM data root (contains {subject_id}/{yyyymmdd}/{series}/001)
ROOT_DIR      = os.path.join(REPO_DIR, "data", "OAI", "1243742", "image03", "12m", "1.E.1")

# Semi-quantitative label file
LABEL_FILE    = os.path.join(REPO_DIR, "data", "OAI", "Package_1243743", "oai_kxrsemiquant01.txt")

# Output directory
OUTPUT_DIR    = os.path.join(REPO_DIR, "data", "OAI", "processed")
IMAGES_DIR    = os.path.join(OUTPUT_DIR, "images")

os.makedirs(IMAGES_DIR, exist_ok=True)
print(f"Output dir : {OUTPUT_DIR}")
print(f"Images dir : {IMAGES_DIR}")

Output dir : /Users/yuhaohe/Documents/RA/healthcare_AI_RA/ConceptBottleneck/data/OAI/processed
Images dir : /Users/yuhaohe/Documents/RA/healthcare_AI_RA/ConceptBottleneck/data/OAI/processed/images


## Step 1 — Find valid (KNEE DICOM, label) pairs

In [2]:
import pydicom
import pandas as pd
from datetime import datetime
from collections import Counter

# ---------- 1a. Walk ROOT_DIR and collect all DICOM 001 files ----------

def find_dicom_files(root):
    """Walk root recursively and return paths to all '001' DICOM files."""
    paths = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname in ("001",):               # OAI DICOM files are named '001'
                paths.append(os.path.join(dirpath, fname))
    return sorted(paths)


all_dicom_paths = find_dicom_files(ROOT_DIR)
print(f"Total DICOM '001' files found: {len(all_dicom_paths)}")

Total DICOM '001' files found: 10


In [3]:
# ---------- 1b. Filter to KNEE images only ----------

def is_knee(dicom_path):
    """Return True if the DICOM BodyPartExamined tag is KNEE."""
    try:
        ds = pydicom.dcmread(dicom_path, stop_before_pixels=True, force=True)
        body_part = ds.get((0x0018, 0x0015), None)
        if body_part is None:
            return False
        return str(body_part.value).upper() == "KNEE"
    except Exception:
        return False


knee_paths = [p for p in all_dicom_paths if is_knee(p)]
print(f"KNEE DICOM files: {len(knee_paths)} / {len(all_dicom_paths)}")

KNEE DICOM files: 8 / 10


In [4]:
# ---------- 1c. Parse subject_id and interview_date from path ----------
# Path structure: ROOT_DIR/{subject_id}/{yyyymmdd}/{series}/001

def parse_path_metadata(dicom_path):
    parts = dicom_path.split(os.sep)
    subject_id     = parts[-4]                             # e.g. '9001104'
    date_yyyymmdd  = parts[-3]                             # e.g. '20060720'
    interview_date = datetime.strptime(date_yyyymmdd, "%Y%m%d").strftime("%m/%d/%Y")
    return subject_id, interview_date


knee_meta = []
for p in knee_paths:
    sid, idate = parse_path_metadata(p)
    knee_meta.append({"dicom_path": p, "src_subject_id": sid, "interview_date": idate})

df_knee = pd.DataFrame(knee_meta)
print(df_knee[["src_subject_id", "interview_date"]].to_string())

  src_subject_id interview_date
0        9000099     07/13/2006
1        9000622     07/10/2006
2        9001104     07/20/2006
3        9002116     08/04/2006
4        9002430     06/01/2006
5        9003126     07/27/2006
6        9003175     06/15/2006
7        9003658     06/22/2006


In [5]:
# ---------- 1d. Load label file and inner-join ----------

# Skip the second header row (row index 1 is a description row)
df_labels = pd.read_csv(LABEL_FILE, sep="\t", skiprows=[1], low_memory=False)

# Normalise types for joining
df_labels["src_subject_id"] = df_labels["src_subject_id"].astype(str).str.strip()
df_labels["interview_date"] = df_labels["interview_date"].astype(str).str.strip()

# Keep only the columns we need at this stage
LABEL_COLS = [
    "src_subject_id", "interview_date", "visit", "readprj", "side",
    "xrkl",
    # 10 kept concepts
    "xrosfm", "xrscfm", "xrjsm", "xrostm", "xrsctm",
    "xrosfl", "xrscfl", "xrjsl", "xrostl", "xrsctl",
    # 8 filtered concepts (kept in CSV for reference)
    "xrcyfm", "xrchm", "xrcytm", "xrattm",
    "xrcyfl", "xrchl", "xrcytl", "xrattl",
    "barcode",
]
df_labels = df_labels[LABEL_COLS].copy()

# Map numeric side codes to strings using the same mapping as NonImageData:
# side_mappings = {1:'right', 2:'left'}  (non_image_data_processing.py line 51)
df_labels["side"] = pd.to_numeric(df_labels["side"], errors="coerce")
df_labels["side"] = df_labels["side"].map({1: "right", 2: "left"})
df_labels = df_labels.dropna(subset=["side"])

# Filter readprj==15 for timepoints 00–48m (load_semiquantitative_xray_data logic)
df_labels["readprj"] = pd.to_numeric(df_labels["readprj"], errors="coerce")
# Recode miscoded project IDs: readprj==42 → 37 (per OAI notes, also in pain-disparities)
df_labels.loc[df_labels["readprj"] == 42, "readprj"] = 37
df_labels = df_labels[df_labels["readprj"] == 15].copy()

# Inner join on (src_subject_id, interview_date)
df_raw = pd.merge(df_knee, df_labels, on=["src_subject_id", "interview_date"], how="inner")

print(f"Valid (DICOM, label) pairs — {len(df_raw)} rows  "
      f"({df_raw['dicom_path'].nunique()} unique DICOMs × 2 sides)")
print(df_raw[["src_subject_id", "interview_date", "side", "xrkl"]].to_string())

Valid (DICOM, label) pairs — 16 rows  (8 unique DICOMs × 2 sides)
   src_subject_id interview_date   side  xrkl
0         9000099     07/13/2006  right   2.0
1         9000099     07/13/2006   left   4.0
2         9000622     07/10/2006  right   1.0
3         9000622     07/10/2006   left   1.0
4         9001104     07/20/2006  right   3.0
5         9001104     07/20/2006   left   1.0
6         9002116     08/04/2006  right   2.0
7         9002116     08/04/2006   left   3.0
8         9002430     06/01/2006  right   2.0
9         9002430     06/01/2006   left   3.0
10        9003126     07/27/2006  right   0.0
11        9003126     07/27/2006   left   0.0
12        9003175     06/15/2006  right   0.0
13        9003175     06/15/2006   left   0.0
14        9003658     06/22/2006  right   0.0
15        9003658     06/22/2006   left   0.0


## Step 2 — Image preprocessing (using pain-disparities `XRayImageDataset` methods)

We import `XRayImageDataset` from the pain-disparities submodule and call its methods directly:
- `get_resized_pixel_array_from_dicom_image()` — resize DICOM to 1024×1024
- `compute_dataset_image_statistics_and_divide_by_max()` — divide by global max, compute mean/std
- `cut_image_in_half()` — split bilateral image into left/right knee (with 1.1× margin + radiological convention flip)
- `make_images_RGB_and_zscore()` — replicate to 3-channel RGB, z-score with dataset-wide stats

`constants_and_util.py` has `assert os.path.exists(...)` guards at module level.
We patch the path variables to real directories before importing to bypass these guards.

In [6]:
import sys
import types

PAIN_DISPARITIES_DIR = os.path.join(REPO_DIR, "submodule", "pain-disparities")

# --- Patch constants_and_util path guards before importing ---
# constants_and_util.py has module-level asserts:
#   assert os.path.exists(INDIVIDUAL_IMAGES_PATH)
#   assert os.path.exists(FITTED_MODEL_DIR)
# We point these to existing directories so the asserts pass.
# We also need FITTED_MODEL_DIR/configs, results, model_weights subdirs.
_dummy_fitted_model_dir = os.path.join(OUTPUT_DIR, "_fitted_models_stub")
for _sub in ["configs", "results", "model_weights"]:
    os.makedirs(os.path.join(_dummy_fitted_model_dir, _sub), exist_ok=True)

# Insert the submodule dir at the front of sys.path so the imports resolve
if PAIN_DISPARITIES_DIR not in sys.path:
    sys.path.insert(0, PAIN_DISPARITIES_DIR)

# Pre-create a stub for constants_and_util so we can set the path values
# before the module-level asserts fire.
import importlib
import importlib.util

# Load constants_and_util source without executing the assert lines by
# temporarily overriding INDIVIDUAL_IMAGES_PATH and FITTED_MODEL_DIR in builtins.
# The cleanest approach: read the file, patch the two placeholder strings,
# write to a temp module, and import that.
import tempfile, shutil

_cu_src_path = os.path.join(PAIN_DISPARITIES_DIR, "constants_and_util.py")
with open(_cu_src_path, "r") as _f:
    _cu_src = _f.read()

# Replace the placeholder strings with real paths before the asserts run
_cu_src = _cu_src.replace(
    "'THIS_IS_A_TEMPORARY_PATH_PLEASE_REPLACE_ME' # points to the directory which stores the processed data",
    repr(IMAGES_DIR)
)
_cu_src = _cu_src.replace(
    "'THIS_IS_A_TEMPORARY_PATH_PLEASE_REPLACE_ME' # This is where you store the fitted models.",
    repr(_dummy_fitted_model_dir)
)
# Also patch the other two placeholders to avoid assert errors when REPROCESS_RAW_DATA is checked
_cu_src = _cu_src.replace(
    "'THIS_IS_A_TEMPORARY_PATH_PLEASE_REPLACE_ME' # Set this path to point to the directory where you downloaded the NON-IMAGE OAI data",
    repr(OUTPUT_DIR)
)
_cu_src = _cu_src.replace(
    "'THIS_IS_A_TEMPORARY_PATH_PLEASE_REPLACE_ME' # Set this path to point to the directory where you downloaded the IMAGE OAI data",
    repr(OUTPUT_DIR)
)

_tmp_dir = tempfile.mkdtemp()
_cu_patched_path = os.path.join(_tmp_dir, "constants_and_util.py")
with open(_cu_patched_path, "w") as _f:
    _f.write(_cu_src)

# Load the patched module under the canonical name
_spec = importlib.util.spec_from_file_location("constants_and_util", _cu_patched_path)
_cu_mod = importlib.util.module_from_spec(_spec)
sys.modules["constants_and_util"] = _cu_mod
_spec.loader.exec_module(_cu_mod)

print("constants_and_util imported successfully.")
print(f"  RESAMPLED_IMAGE_SIZE = {_cu_mod.RESAMPLED_IMAGE_SIZE}")
print(f"  INDIVIDUAL_IMAGES_PATH = {_cu_mod.INDIVIDUAL_IMAGES_PATH}")

# Now import image_processing (it does 'from constants_and_util import *')
import image_processing
from image_processing import XRayImageDataset
print("image_processing imported successfully.")

SyntaxError: invalid syntax (constants_and_util.py, line 72)

In [ ]:
import numpy as np

# ---------- 2a. Build a minimal XRayImageDataset-like object ----------
# XRayImageDataset.__init__ expects OAI directory structure we don't have.
# We instantiate a bare object and call the methods directly,
# mirroring how pain-disparities processes data internally.

xray_ds = object.__new__(XRayImageDataset)

# Set the attributes that the methods we call depend on:
#   extra_margin_for_each_image  — used by cut_image_in_half()
#   diacom_image_statistics      — populated by compute_dataset_image_statistics_and_divide_by_max()
#   normalization_method         — used by make_images_RGB_and_zscore()
#   show_both_knees_in_each_image, crop_to_just_the_knee, downsample_factor_on_reload
#                                — shape assertions in make_images_RGB_and_zscore()
xray_ds.extra_margin_for_each_image = 1.1          # matches pain-disparities default
xray_ds.diacom_image_statistics     = {}
xray_ds.normalization_method        = 'our_statistics'
xray_ds.show_both_knees_in_each_image = False
xray_ds.crop_to_just_the_knee         = False
xray_ds.downsample_factor_on_reload   = None
xray_ds.images                        = []

print("Minimal XRayImageDataset proxy ready.")

In [ ]:
# ---------- 2b. Load and resize all unique DICOMs ----------
# Uses XRayImageDataset.get_resized_pixel_array_from_dicom_image()
# which calls cv2.resize to (1024, 1024) — same as RESAMPLED_IMAGE_SIZE.

unique_dicoms = df_raw["dicom_path"].unique()
print(f"Unique DICOMs to process: {len(unique_dicoms)}")

# Store in the images list format that pain-disparities methods expect:
#   self.images[i]['unnormalized_image_array']  — raw resized float array
#   self.images[i]['subject_id']               — for bookkeeping
xray_ds.images = []
for dicom_path in unique_dicoms:
    ds = pydicom.dcmread(dicom_path)
    # get_resized_pixel_array_from_dicom_image returns a float64 1024×1024 array
    arr = xray_ds.get_resized_pixel_array_from_dicom_image(ds)
    sid, _ = parse_path_metadata(dicom_path)
    xray_ds.images.append({
        'unnormalized_image_array': arr.astype(np.float32),
        'image_array_scaled_to_zero_one': None,
        'left_knee_scaled_to_zero_one': None,
        'right_knee_scaled_to_zero_one': None,
        'subject_id': sid,
        'dicom_path': dicom_path,
    })

print(f"Loaded and resized {len(xray_ds.images)} DICOMs.")
print(f"Example shape: {xray_ds.images[0]['unnormalized_image_array'].shape}")

In [ ]:
# ---------- 2c. Compute dataset statistics and divide by global max ----------
# Calls XRayImageDataset.compute_dataset_image_statistics_and_divide_by_max()
# which:
#   1. Finds global max across all unnormalized_image_array values
#   2. Divides each image by that max → image_array_scaled_to_zero_one in [0,1]
#   3. Stores max, mean, std in self.diacom_image_statistics

xray_ds.compute_dataset_image_statistics_and_divide_by_max(just_normalize_cropped_knees=False)

print("\nDataset image statistics:")
for k, v in xray_ds.diacom_image_statistics.items():
    print(f"  {k}: {v:.6f}")

In [ ]:
# ---------- 2d. Cut each bilateral image into left and right knee ----------
# Calls XRayImageDataset.cut_images_in_two() which loops over self.images and
# calls cut_image_in_half() for each.
#
# cut_image_in_half() details (from pain-disparities):
#   border_left  = int(512 * 1.1) = 563
#   border_right = 1024 - 563 = 461
#   image_on_left  = image[:, :563]  then flipped  → right_knee (patient's right)
#   image_on_right = image[:, 461:]               → left_knee  (patient's left)
#
# IMPORTANT: radiological convention — right side of film = patient's LEFT knee.

xray_ds.cut_images_in_two()

print(f"Cut complete. Left/right knee shapes for image 0:")
print(f"  left  knee: {xray_ds.images[0]['left_knee_scaled_to_zero_one'].shape}")
print(f"  right knee: {xray_ds.images[0]['right_knee_scaled_to_zero_one'].shape}")

In [ ]:
# ---------- 2e. Convert to RGB and z-score ----------
# Calls XRayImageDataset.make_images_RGB_and_zscore() which:
#   1. Replicates each half-image grayscale to (3, H, W)
#   2. Z-scores using dataset-wide mean and std from diacom_image_statistics
#      (mean_of_zero_one_data_full_image, std_of_zero_one_data_full_image)
# Result is stored in self.images[i]['left_knee'] and ['right_knee'].

xray_ds.make_images_RGB_and_zscore()

print(f"RGB+zscore complete. Shape for image 0, left knee: {xray_ds.images[0]['left_knee'].shape}")
print(f"  min={xray_ds.images[0]['left_knee'].min():.4f}  max={xray_ds.images[0]['left_knee'].max():.4f}")

In [ ]:
# ---------- 2f. Save .npy files; skip if already present (re-runnable) ----------

import json

# Save normalisation stats for reproducibility
stats = {
    "global_max":  xray_ds.diacom_image_statistics["max_full_image"],
    "global_mean": xray_ds.diacom_image_statistics["mean_of_zero_one_data_full_image"],
    "global_std":  xray_ds.diacom_image_statistics["std_of_zero_one_data_full_image"],
}
with open(os.path.join(OUTPUT_DIR, "stats.json"), "w") as f:
    json.dump(stats, f, indent=2)
print("stats.json saved.")

saved_count   = 0
skipped_count = 0

for img_dict in xray_ds.images:
    sid = img_dict["subject_id"]
    for side in ["left", "right"]:
        out_path = os.path.join(IMAGES_DIR, f"{sid}_{side}.npy")
        if os.path.exists(out_path):
            skipped_count += 1
            continue
        np.save(out_path, img_dict[f"{side}_knee"].astype(np.float32))
        saved_count += 1

print(f"Images saved  : {saved_count}")
print(f"Images skipped (already exist): {skipped_count}")

## Step 3 — Label processing (using `NonImageData.load_semiquantitative_xray_data()` logic)

We replicate the key logic from `NonImageData.load_semiquantitative_xray_data()` in pain-disparities:
1. **readprj filter**: use `readprj==15` for timepoints 00–48m (already done in Step 1d)
2. **readprj recode**: `readprj==42 → 37` (already done in Step 1d)
3. **Fill concepts=0**: for participants who never had `xrkl >= 2` at current or prior timepoints,
   set all non-JSN/non-KLG concept columns to 0 where missing
4. **Truncate JSN**: floor fractional grades (1.2→1, 1.4→1 etc.)
5. **Merge KLG**: 0+1→0, then shift → 4-level label (0–3)

In [ ]:
# ---------- 3a. Numeric coercion ----------

CONCEPT_COLS_KEPT = [
    "xrosfm", "xrscfm", "xrjsm", "xrostm", "xrsctm",
    "xrosfl", "xrscfl", "xrjsl", "xrostl", "xrsctl",
]
CONCEPT_COLS_FILTERED = [
    "xrcyfm", "xrchm", "xrcytm", "xrattm",
    "xrcyfl", "xrchl", "xrcytl", "xrattl",
]
ALL_CONCEPT_COLS = CONCEPT_COLS_KEPT + CONCEPT_COLS_FILTERED
JSN_COLS = ["xrjsm", "xrjsl"]

df = df_raw.copy()

df["xrkl"] = pd.to_numeric(df["xrkl"], errors="coerce")
for c in ALL_CONCEPT_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Numeric coercion done. NaN counts:")
print(df[["xrkl"] + ALL_CONCEPT_COLS].isnull().sum())

In [ ]:
# ---------- 3b. Truncate fractional JSN grades ----------
# From load_semiquantitative_xray_data(): JSN (xrjsl, xrjsm) grades can be
# fractional (e.g. 1.2, 1.4) to indicate temporal progression within a grade.
# These are NOT fractional severity — floor to integer.
# (pain-disparities does this implicitly by treating them as ordinal integers.)

for c in JSN_COLS:
    df[c] = np.floor(pd.to_numeric(df[c], errors="coerce").fillna(0)).astype("Int64")

print("JSN columns truncated.")
print(df[JSN_COLS].value_counts().head(10))

In [ ]:
# ---------- 3c. Fill non-JSN/non-KLG concepts to 0 for never-KLG>=2 participants ----------
# From NonImageData.load_semiquantitative_xray_data() (pain-disparities):
#
#   "Scores for other IRFs (osteophytes, subchondral sclerosis, cysts and attrition)
#    are available only in participants with definite radiographic OA at least one knee
#    at one (or more) of the time points."
#
# pain-disparities logic:
#   participants_who_have_had_definite_radiographic_oa = set of IDs with xrkl >= 2
#     at any timepoint <= current timepoint.
#   For missing values in non-JSN/non-KLG concept cols:
#     if participant NOT in that set → fill with 0
#     if participant IS  in that set → drop the row (truly missing scored data)
#
# With our single timepoint we approximate:
#   participants_who_have_had_definite_radiographic_oa = those with xrkl >= 2 in ANY
#   row of df (i.e., either knee at this visit).
# When you add more timepoints, revisit this to check across all prior timepoints.

sids_ever_oa = set(df.loc[df["xrkl"] >= 2, "src_subject_id"])
print(f"Participants with definite radiographic OA (KLG >= 2) in this dataset: {len(sids_ever_oa)}")

FILL_COLS = [c for c in ALL_CONCEPT_COLS if c not in JSN_COLS]

for c in FILL_COLS:
    missing_mask       = df[c].isnull()
    should_have_data   = df["src_subject_id"].isin(sids_ever_oa)
    # fill missing for those who never had KLG >= 2
    fill_mask = missing_mask & (~should_have_data)
    df.loc[fill_mask, c] = 0
    if fill_mask.sum() > 0:
        print(f"  {c}: filled {fill_mask.sum()} missing values with 0")

# Drop rows where a participant should have concept scores (had KLG >= 2)
# but is still missing them — mirrors pain-disparities behavior.
rows_to_drop = pd.Series(False, index=df.index)
for c in FILL_COLS:
    missing_mask     = df[c].isnull()
    should_have_data = df["src_subject_id"].isin(sids_ever_oa)
    rows_to_drop = rows_to_drop | (missing_mask & should_have_data)

n_dropped = rows_to_drop.sum()
if n_dropped > 0:
    print(f"\nDropping {n_dropped} rows where participant should have concept scores but doesn't.")
    df = df[~rows_to_drop].copy()
else:
    print("No rows dropped for missing concept scores.")

# Also drop rows missing xrkl (mirrors pain-disparities dropna on xrkl)
before = len(df)
df = df.dropna(subset=["xrkl"]).copy()
print(f"Dropped {before - len(df)} rows missing xrkl. Remaining: {len(df)}")

In [ ]:
# ---------- 3d. Merge KLG classes: 0+1 → 0, then shift down → 4-level (0-3) ----------
# This is the target label used in the CBM paper.
# Mirrors the label construction in pain-disparities / CBM pipeline.

def merge_klg(klg):
    """Map raw KLG (0-4) to merged KLG (0-3): 0→0, 1→0, 2→1, 3→2, 4→3."""
    if pd.isnull(klg):
        return None
    klg = int(klg)
    if klg <= 1:
        return 0
    return klg - 1

df["xrkl_merged"] = df["xrkl"].map(merge_klg)

print("KLG distribution (raw):")
print(df["xrkl"].value_counts().sort_index())
print("\nKLG distribution (merged):")
print(df["xrkl_merged"].value_counts().sort_index())

## Step 4 — Build and save manifest

In [ ]:
# ---------- 4a. Add processed_image_path column ----------

def make_image_path(row):
    return os.path.join(IMAGES_DIR, f"{row['src_subject_id']}_{row['side']}.npy")

df["processed_image_path"] = df.apply(make_image_path, axis=1)

# Warn if any .npy file is missing
missing = df["processed_image_path"].map(lambda p: not os.path.exists(p))
if missing.any():
    print(f"WARNING: {missing.sum()} processed image files not found:")
    print(df.loc[missing, "processed_image_path"].tolist())
else:
    print("All processed image .npy files exist.")

In [ ]:
# ---------- 4b. Assemble final manifest column order ----------

MANIFEST_COLS = [
    # identifiers
    "src_subject_id", "interview_date", "visit", "side", "readprj", "barcode",
    # file paths
    "dicom_path", "processed_image_path",
    # primary label
    "xrkl", "xrkl_merged",
    # 10 kept concept labels
    "xrosfm", "xrscfm", "xrjsm", "xrostm", "xrsctm",
    "xrosfl", "xrscfl", "xrjsl", "xrostl", "xrsctl",
    # 8 filtered concept labels (kept for reference)
    "xrcyfm", "xrchm", "xrcytm", "xrattm",
    "xrcyfl", "xrchl", "xrcytl", "xrattl",
]

manifest = df[MANIFEST_COLS].copy()
manifest = manifest.reset_index(drop=True)

manifest_path = os.path.join(OUTPUT_DIR, "manifest.csv")
manifest.to_csv(manifest_path, index=False)
print(f"Manifest saved: {manifest_path}")
print(f"Total rows   : {len(manifest)}")
manifest.head()

## Step 5 — Sanity checks

In [ ]:
# ---------- 5a. Basic stats ----------

print("=" * 50)
print(f"Unique subjects  : {manifest['src_subject_id'].nunique()}")
print(f"Total (X, y) pairs (rows): {len(manifest)}")
print(f"  left  knees: {(manifest['side'] == 'left').sum()}")
print(f"  right knees: {(manifest['side'] == 'right').sum()}")
print("\nKLG (merged) distribution:")
print(manifest["xrkl_merged"].value_counts().sort_index())
print("\nNaN counts in label columns:")
print(manifest[["xrkl", "xrkl_merged"] + CONCEPT_COLS_KEPT].isnull().sum())
print("=" * 50)

In [ ]:
# ---------- 5b. Visual check: plot one image per side for first subject ----------

import matplotlib.pyplot as plt

first_subject = manifest["src_subject_id"].iloc[0]
rows_for_subject = manifest[manifest["src_subject_id"] == first_subject]

fig, axes = plt.subplots(1, len(rows_for_subject), figsize=(6 * len(rows_for_subject), 6))
if len(rows_for_subject) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, rows_for_subject.iterrows()):
    img = np.load(row["processed_image_path"])   # shape (3, H, W)
    ax.imshow(img[0], cmap="gray")
    ax.set_title(
        f"Subject {row['src_subject_id']}\n"
        f"side={row['side']}  KLG={row['xrkl']} (merged={row['xrkl_merged']})"
    )
    ax.axis("off")

plt.suptitle("Processed knee half-images (z-scored, channel 0 shown)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ---------- 5c. Print manifest summary ----------

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 40)
print("Manifest (all rows):")
manifest